# URA Conversational AI (CSV + PDF → RAG + TTS)
- Web: Gemma-2B or Llama-3.2-3B (high accuracy)
- Mobile/offline: Llama-3.2-1B (speed/balance)
- Background: Flan-T5-Small for auto-tagging / utility tasks
- Speech: TTS for English and Luganda via multilingual model

In [ ]:
# Install dependencies (run once per environment)
%pip install -q -U langchain langchain-community sentence-transformers pymupdf4llm faiss-cpu datasets transformers accelerate evaluate torch TTS pydub

In [ ]:
import os, json, random, pathlib
import pandas as pd
from datasets import Dataset, DatasetDict
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import pymupdf4llm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, pipeline
from transformers.trainer_utils import IntervalStrategy
import evaluate

PROJECT_ROOT = pathlib.Path('/workspaces/FinalYearProject')
DATASETS_DIR = PROJECT_ROOT / 'datasets'
PDF_DIR = PROJECT_ROOT / 'pdfs'
TTT_DIR = PROJECT_ROOT / 'TTT'
LGAUDIO_DIR = PROJECT_ROOT / 'lgaudio'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GEN_MODELS = {
    'web_high_accuracy': 'google/gemma-2-2b-it',
    'web_alt': 'meta-llama/Llama-3.2-3B-Instruct',
    'mobile_offline': 'meta-llama/Llama-3.2-1B-Instruct',
    'background_t5': 'google/flan-t5-small',
}

EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
random.seed(42)

In [ ]:
csv_files = sorted(DATASETS_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files in {DATASETS_DIR}')
stats = []
for path in csv_files:
    try:
        df = pd.read_csv(path)
        stats.append({'file': path.name, 'rows': len(df), 'cols': list(df.columns)})
    except Exception as exc:
        stats.append({'file': path.name, 'error': str(exc)})
stats_df = pd.DataFrame(stats)
stats_df.head(10)

In [ ]:
if csv_files:
    sample_df = pd.read_csv(csv_files[0]).head(5)
    sample_df
else:
    print('No CSV files found; add data to datasets/.')

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ''
    txt = str(text).replace('\n', ' ').replace('\r', ' ')
    return ' '.join(txt.split())

In [ ]:
question_candidates = {'question', 'questions', 'q'}
answer_candidates = {'answer', 'answers', 'a', 'response', 'resp'}
frames = []
for path in csv_files:
    df = pd.read_csv(path)
    columns_lower = {c.lower(): c for c in df.columns}
    q_col = next((columns_lower[c] for c in columns_lower if c in question_candidates), df.columns[0])
    a_col = next((columns_lower[c] for c in columns_lower if c in answer_candidates and columns_lower[c] != q_col), df.columns[-1])
    df = df[[q_col, a_col]].rename(columns={q_col: 'question', a_col: 'answer'})
    df['question'] = df['question'].apply(clean_text)
    df['answer'] = df['answer'].apply(clean_text)
    df['context'] = df['answer']
    df['tag'] = path.stem.replace('_', ' ')
    df['source'] = path.name
    frames.append(df)
qa_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['question', 'answer', 'context', 'tag', 'source'])
qa_df.head(3)

In [ ]:
# Preprocess QA rows (length filters, empties, duplicates)
def preprocess_qa(df, min_q_words=3, min_a_words=3, max_words=512):
    if df.empty:
        return df, {}
    work = df.copy()
    work['question'] = work['question'].fillna('').str.strip()
    work['answer'] = work['answer'].fillna('').str.strip()
    work['context'] = work['context'].fillna('').str.strip()
    mask_nonempty = (work['question'] != '') & (work['answer'] != '')
    work = work[mask_nonempty]
    work['q_words'] = work['question'].str.split().apply(len)
    work['a_words'] = work['answer'].str.split().apply(len)
    work['ctx_words'] = work['context'].str.split().apply(len)
    work = work[(work['q_words'] >= min_q_words) & (work['a_words'] >= min_a_words)]
    work = work[(work['q_words'] <= max_words) & (work['a_words'] <= max_words)]
    before_dedup = len(work)
    work = work.drop_duplicates(subset=['question', 'answer'])
    stats = {
        'rows_before': int(len(df)),
        'rows_after': int(len(work)),
        'dropped_empty_q_or_a': int(len(df) - len(df[mask_nonempty])),
        'dropped_too_short_or_long': int(len(df[mask_nonempty]) - before_dedup),
        'dropped_duplicates': int(before_dedup - len(work)),
    }
    work = work.drop(columns=['q_words', 'a_words', 'ctx_words'])
    return work.reset_index(drop=True), stats

qa_df, qa_stats = preprocess_qa(qa_df)
print('QA preprocessing stats:', qa_stats)

In [ ]:
pdf_chunks = []
for pdf_path in sorted(PDF_DIR.glob('*.pdf')):
    try:
        md_text = pymupdf4llm.to_markdown(pdf_path)
        pdf_chunks.append({'source': pdf_path.name, 'text': clean_text(md_text)})
    except Exception as exc:
        print(f'Failed to read {pdf_path.name}: {exc}')
print(f'Loaded {len(pdf_chunks)} PDFs from {PDF_DIR}')

In [ ]:
# Quick EDA for loaded QA/PDF data
if qa_df.empty:
    print('No QA rows loaded; check datasets/.')
else:
    print(f'QA rows: {len(qa_df)}, columns: {list(qa_df.columns)}')
    print('\nTop sources (rows):')
    print(qa_df['source'].value_counts().head(10))
    print('\nTop tags:')
    print(qa_df['tag'].value_counts().head(10))
    lengths = qa_df['context'].str.split().apply(len)
    print('\nContext length (words) summary:')
    print(lengths.describe(percentiles=[0.5, 0.9, 0.95]))
    missing = qa_df.isna().mean()
    print('\nMissing fraction per column:')
    print(missing)
    sample = qa_df.sample(min(3, len(qa_df)), random_state=42)[['question', 'answer', 'tag', 'source']]
    display(sample)

print(f'Loaded {len(pdf_chunks)} PDFs from {PDF_DIR}')
if pdf_chunks:
    pdf_lengths = pd.Series([len(ch['text'].split()) for ch in pdf_chunks])
    print('\nPDF text length (words) summary:')
    print(pdf_lengths.describe(percentiles=[0.5, 0.9, 0.95]))
    print('\nSample PDF entry:')
    print(pdf_chunks[0]['source'])
    print(pdf_chunks[0]['text'][:400] + '...')

In [ ]:
corpus = []
for row in qa_df.itertuples():
    corpus.append({'text': f"Question: {row.question}\nAnswer: {row.answer}", 'metadata': {'source': row.source, 'tag': row.tag}})
for chunk in pdf_chunks:
    corpus.append({'text': chunk['text'], 'metadata': {'source': chunk['source'], 'tag': 'pdf'}})
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
docs = [Document(page_content=item['text'], metadata=item['metadata']) for item in corpus]
split_docs = splitter.split_documents(docs)
len(split_docs)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
vectordb = FAISS.from_documents(split_docs, embeddings) if split_docs else None
if vectordb:
    vectordb.save_local(str(OUTPUT_DIR / 'faiss_index'))
vectordb

In [ ]:
def retrieve(query, top_k=5):
    if vectordb is None:
        return []
    return vectordb.similarity_search(query, k=top_k)

retrieve('How do I pay taxes?', 3)

In [ ]:
def load_text_generator(target='web_high_accuracy'):
    model_id = GEN_MODELS[target]
    if target == 'background_t5':
        tok = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
        return pipeline('text2text-generation', model=model, tokenizer=tok, device_map='auto')
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map='auto')
    return pipeline('text-generation', model=model, tokenizer=tok, device_map='auto', max_new_tokens=256, temperature=0.2)

In [ ]:
def generate_answer(query, lang_hint='en', top_k=4, target='web_high_accuracy'):
    docs = retrieve(query, top_k)
    context = '\n\n'.join([d.page_content for d in docs])
    prompt = (
        'You are a URA customer-service assistant. ' 
        f'Answer in {lang_hint} (English=\"en\", Luganda=\"lg\"). ' 
        'Be concise (<=120 words) and cite policy when present.\n' 
        f'Question: {query}\nContext: {context}\nAnswer:'
    )
    text_gen = load_text_generator(target)
    result = text_gen(prompt)[0]['generated_text']
    return result

# Example (large downloads; run on GPU only):
# generate_answer('How do I get a TIN?', lang_hint='lg', target='mobile_offline')

In [ ]:
if not qa_df.empty:
    tag_dataset = Dataset.from_pandas(qa_df[['question', 'context', 'tag']])
    tag_dataset = tag_dataset.train_test_split(test_size=0.1, seed=42)
    print(tag_dataset)
else:
    tag_dataset = None
    print('No QA data to tag yet.')

In [ ]:
if tag_dataset:
    t5_model_name = GEN_MODELS['background_t5']
    t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name)
    t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name)

    def preprocess_tag(batch):
        inputs = [f'ticket: {q} context: {c}' for q, c in zip(batch['question'], batch['context'])]
        model_inputs = t5_tokenizer(inputs, max_length=512, truncation=True)
        labels = t5_tokenizer(batch['tag'], max_length=32, truncation=True)
        model_inputs['labels'] = labels['input_ids']
        return model_inputs

    tokenized_tags = tag_dataset.map(preprocess_tag, batched=True)
    data_collator = DataCollatorForSeq2Seq(t5_tokenizer, model=t5_model)
    metric = evaluate.load('accuracy')

    def compute_tag_metrics(eval_pred):
        preds, labels = eval_pred
        preds = t5_tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = t5_tokenizer.batch_decode([[l for l in label if l != -100] for label in labels], skip_special_tokens=True)
        return {'accuracy': metric.compute(predictions=preds, references=labels)['accuracy']}

    train_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / 'tagger'),
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=5e-5,
        num_train_epochs=1,
        evaluation_strategy=IntervalStrategy.EPOCH,
        predict_with_generate=True,
        fp16=False,
        logging_steps=25,
        save_strategy=IntervalStrategy.NO,
    )

    tagger = Trainer(
        model=t5_model,
        args=train_args,
        train_dataset=tokenized_tags['train'],
        eval_dataset=tokenized_tags['test'],
        tokenizer=t5_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_tag_metrics,
    )

    # tagger.train()
else:
    print('Tagger not initialized; no data available.')

In [ ]:
from TTS.api import TTS

tts_model = 'tts_models/multilingual/multi-dataset/xtts_v2'
tts = TTS(tts_model)

def speak(text, language='en', file_name='tts_output.wav'):
    out_path = OUTPUT_DIR / file_name
    tts.tts_to_file(text=text, file_path=out_path, language=language)
    return out_path

# speak('Webale nnyo!', language='lg', file_name='luganda.wav')
# speak('Thank you for contacting URA.', language='en', file_name='english.wav')

## Next steps
- Run installs, then execute cells sequentially to materialize the index and tagger artifacts in artifacts/.
- Swap model ids with quantized/gguf variants for mobile/offline targets (llama.cpp).
- Consider distilling Gemma/Llama responses into the 1B mobile model for latency gains.
- Add ASR front-end if you want full speech-to-speech with the TTS block above.